# Multi-Bearing Full Life-Cycle Evaluation
This notebook visualizes the anomaly scores (MSE) of a single trained model across multiple bearing datasets (B01, B02, B03, B04, B05). 
It helps in understanding how well the model generalizes to different bearings and tracks their degradation trends.

In [25]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
import pandas as pd
from tqdm.auto import tqdm

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.models.mamba import MambaTSOfficial, MambaTSConfig, HybridMambaCNN
from src.data import BearingDataset
from src.evaluation.anomaly_scorer import calculate_anomaly_score
from src.evaluation.metrics import calculate_threshold_pot

In [26]:
# --- Configuration ---
model_path = os.path.join(project_root, "results/models/mamba1_hybrid_best.pth")
config_path = os.path.join(project_root, "configs/default.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

lookback = config['data'].get('lookback', 4096)
horizon = config['data'].get('horizon', 1024)
patch_size = config['data'].get('patch_size', 2048)
stride = config['data'].get('stride', 1024)
sampling_rate = config['data'].get('sampling_rate', 128000)
highpass_freq = config['data'].get('highpass_freq', 2000)

SKIP_RATIO = 0  # Standard skip_ratio for the dataset
POT_Q = 1e-3       # POT probability parameter

print(f"Device: {device}")
print(f"Model Path: {model_path}")
print(f"Skip ratio: {SKIP_RATIO*100}%")

Device: cuda
Model Path: /mnt/f/APPS_PJ/mamba-forecast-ad/results/models/mamba1_hybrid_best.pth
Skip ratio: 0%


In [27]:
# --- Load Model ---
def load_model(path, config, device):
    if "hybrid" in path.lower():
        model = HybridMambaCNN({
            'model': {
                'mamba_version': 1,
                'mamba_d_model': config['model'].get('mamba_d_model', 64), 
                'mamba_n_layer': config['model'].get('mamba_n_layer', 4),
                'mamba_d_state': config['model'].get('mamba_d_state', 16), 
                'mamba_d_conv': config['model'].get('mamba_d_conv', 4), 
                'mamba_expand': config['model'].get('mamba_expand', 2),
                'forecast_len': horizon, 
                'patch_size': patch_size, 
                'stride': stride,
                'in_channels': 2, 'lookback': lookback,
                'decomp_kernel': config['model'].get('decomp_kernel', 25), 
                'use_multiscale': True,
            },
            'data': {'patch_size': patch_size, 'stride': stride, 'lookback': lookback}
        })
    else:
        mamba_config = MambaTSConfig(
            in_channels=2, lookback=lookback, forecast_len=horizon,
            patch_size=config['data'].get('patch_size', 64), 
            stride=config['data'].get('stride', 32), 
            d_model=config['model'].get('mamba_d_model', 64), 
            n_layers=config['model'].get('mamba_n_layer', 4)
        )
        model = MambaTSOfficial(mamba_config)
        
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        print("Model loaded successfully.")
    else:
        print(f"WARNING: Model path {path} not found! Using uninitialized model.")
        
    model.to(device)
    model.eval()
    return model

model = load_model(model_path, config, device)

Model loaded successfully.


In [28]:
# --- Evaluation Function ---
def evaluate_bearing(bearing_id, model, device):
    processed_dir = os.path.join(project_root, "data/processed", bearing_id)
    if not os.path.exists(processed_dir):
        print(f"Skip {bearing_id}: path not found.")
        return None
    
    # Initialize dataset with the specific skip_ratio
    dataset = BearingDataset(
        processed_dir, lookback, horizon, stride, split='test', 
        normalize=False, skip_ratio=SKIP_RATIO, train_ratio=0.0
    )
    
    # Although dataset handles samples, we iterate files manually for specific windowing
    # BearingDataset.files is the full list, so we slice it based on its own skip_ratio
    all_files = dataset.files
    n_total = len(all_files)
    start_idx = int(n_total * dataset.skip_ratio)
    eval_files = all_files[start_idx:]
    
    results = {
        'indices': [],
        'mse': [],
        'rms': []
    }
    
    print(f"Processing {bearing_id} ({len(eval_files)}/{n_total} files, skip_ratio={dataset.skip_ratio})...")
    with torch.no_grad():
        for i, f_name in enumerate(tqdm(eval_files, desc=bearing_id)):
            actual_idx = i + start_idx
            f_path = os.path.join(processed_dir, f_name)
            signal = torch.load(f_path, map_location='cpu', weights_only=True)
            
            # Filter if needed
            if highpass_freq > 0:
                from scipy import signal as scipy_signal
                nyq = 0.5 * sampling_rate
                normal_cutoff = highpass_freq / nyq
                b, a = scipy_signal.butter(4, normal_cutoff, btype='high', analog=False)
                sig_filtered = scipy_signal.lfilter(b, a, signal.numpy(), axis=1)
                signal = torch.from_numpy(sig_filtered.copy()).float()
            
            # Extract 3 windows per file for speed
            win_starts = np.linspace(0, signal.shape[1] - lookback - horizon, 3, dtype=int)
            file_errs = []
            
            for start in win_starts:
                x = signal[:, start:start+lookback]
                y = signal[:, start+lookback:start+lookback+horizon]
                
                x_gpu = x.unsqueeze(0).to(device)
                y_gpu = y.unsqueeze(0).to(device)
                
                # Stats
                mean = x.mean(dim=-1, keepdim=True)
                std = x.std(dim=-1, keepdim=True)
                rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True))
                peak = torch.max(torch.abs(x), dim=-1, keepdim=True)[0]
                z = (x - mean) / (std + 1e-8)
                skew = torch.mean(z**3, dim=-1, keepdim=True)
                kurt = torch.mean(z**4, dim=-1, keepdim=True)
                crest = peak / (rms + 1e-8)
                shape = rms / (torch.mean(torch.abs(x), dim=-1, keepdim=True) + 1e-8)
                
                stats = torch.cat([mean, std, rms, peak, skew, kurt, crest, shape], dim=-1).unsqueeze(0).to(device)
                
                y_pred = model(x_gpu, stats)
                mse = calculate_anomaly_score(y_gpu, y_pred, metric='mse', normalized=False).item()
                file_errs.append(mse)
            
            results['indices'].append(actual_idx)
            results['mse'].append(np.mean(file_errs))
            results['rms'].append(dataset.file_rms[f_name])
            
    return results

In [ ]:
# --- Run Loop ---
bearing_ids = ['B01', 'B02', 'B03', 'B04', 'B05']
# bearing_ids = ['B01', 'B04']
all_results = {}

for b_id in bearing_ids:
    res = evaluate_bearing(b_id, model, device)
    if res:
        all_results[b_id] = res

Processing B01 (377/377 files, skip_ratio=0)...


B01:   0%|          | 0/377 [00:00<?, ?it/s]

Processing B02 (1116/1116 files, skip_ratio=0)...


B02:   0%|          | 0/1116 [00:00<?, ?it/s]

In [ ]:
# --- Plotting ---
n_bearings = len(all_results)
if n_bearings > 0:
    fig, axes = plt.subplots(n_bearings, 1, figsize=(15, 5 * n_bearings), sharex=False)
    if n_bearings == 1: axes = [axes]
    
    for i, b_id in enumerate(all_results.keys()):
        res = all_results[b_id]
        ax = axes[i]
        
        # Calculate POT Threshold based on the first 20% of scores (assuming healthy)
        scores = np.array(res['mse'])
        n_init = max(5, int(len(scores) * 0.2))
        healthy_init = scores[:n_init]
        threshold = calculate_threshold_pot(healthy_init, q=POT_Q)
        
        ax.set_title(f"Full Life-cycle: {b_id} (POT Threshold: {threshold:.2e})", fontsize=14)
        
        # RMS on left axis
        color_rms = 'tab:blue'
        ax.plot(res['indices'], res['rms'], color=color_rms, label='RMS')
        ax.set_ylabel('RMS', color=color_rms)
        ax.tick_params(axis='y', labelcolor=color_rms)
        
        # MSE on right axis
        ax_twin = ax.twinx()
        color_mse = 'tab:red'
        ax_twin.plot(res['indices'], res['mse'], color=color_mse, alpha=0.6, label='Anomaly Score (MSE)')
        ax_twin.axhline(y=threshold, color='black', linestyle='--', linewidth=2, label=f'POT Threshold (q={POT_Q})')
        ax_twin.set_ylabel('MSE', color=color_mse)
        ax_twin.tick_params(axis='y', labelcolor=color_mse)
        ax_twin.set_yscale('log')
        
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('File Index')
        ax_twin.legend(loc='upper left')
        
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")